# Test models on held-out segments


In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import your
from tqdm.auto import tqdm


## Set input and output paths


In [2]:
filterbank_file_path = Path('/hercules/results/akazantsev/filterbank_files/B0531+21/B0531+21_59000_48386.fil')
segment_labels_path = Path('/u/akazantsev/ml_dl_for_rfi_mitigation/segments_index_B0531+21_59000_48386.csv')
run_dir = Path('/u/akazantsev/ml_dl_for_rfi_mitigation/runs/punch_rfi_reproduction')
base_name = 'B0531+21_59000_48386'

training_metadata_path = run_dir / f'{base_name}_training_metadata.csv'
cnn_checkpoint_path = run_dir / 'best_model_by_acc.pt'
statistical_model_path = run_dir / 'best_statistical_model.joblib'
segment_metrics_path = run_dir / 'global_test_segment_metrics.csv'
summary_path = run_dir / 'global_test_summary.csv'
replace_existing_outputs = True

for input_path in (
    filterbank_file_path, segment_labels_path, training_metadata_path,
    cnn_checkpoint_path, statistical_model_path
):
    if not input_path.is_file():
        raise FileNotFoundError(f'Input file not found: {input_path}')

for output_path in (segment_metrics_path, summary_path):
    if output_path.exists() and not replace_existing_outputs:
        raise FileExistsError(f'Global-test output already exists: {output_path}')


## Define annotations


In [3]:
n_channels = 256
n_time_samples = 256

NBRFI = 'NBRFI'
BGN = 'BGN'
BGN_RFISEG = 'BGN_RFISEG'

rfi_blocks = {
    1: (204, 1527), 2: (2023, 4436), 3: (4437, 4953),
    4: (6701, 7631), 5: (42546, 45015), 6: (45622, 45705),
    7: (46186, 46927), 8: (46928, 47303), 9: (48686, 48831),
    10: (48832, 49591)
}

segments_rfi_indexes = {
    1: [[230, 235], [239, 241]], 2: [230, 256],
    3: [230, 256], 4: [[230, 235], [239, 243]],
    5: [[189, 195], [199, 202], [205, 209]],
    6: [[189, 195], [199, 202], [206, 210]],
    7: [[190, 193], [199, 202], [206, 210]],
    8: [[188, 194], [198, 203], [206, 210]],
    9: [[188, 194], [198, 203], [206, 210]],
    10: [[188, 194], [198, 203], [206, 210]]
}


def get_rfi_block_id(segment_index):
    for block_id, (start, stop) in rfi_blocks.items():
        if start <= segment_index <= stop:
            return block_id
    return None


def get_rfi_channel_ranges(block_id):
    ranges = segments_rfi_indexes[block_id]
    if isinstance(ranges[0], (list, tuple, np.ndarray)):
        return [(int(start), int(stop)) for start, stop in ranges]
    return [(int(ranges[0]), int(ranges[1]))]


def build_rfi_mask(channel_ranges):
    mask = np.zeros(n_channels, dtype=bool)
    for start, stop in channel_ranges:
        start = max(0, start)
        stop = min(n_channels - 1, stop)
        if start <= stop:
            mask[start:stop + 1] = True
    return mask


## Load held-out segments


In [4]:
segment_labels = pd.read_csv(segment_labels_path)
required_segment_columns = {'global_index', 'segment_index', 'label'}
if not required_segment_columns.issubset(segment_labels.columns):
    raise ValueError('Segment labels have an unexpected schema')

clean_segments = segment_labels.loc[
    segment_labels['label'].isna(), ['global_index', 'segment_index']
].copy()
clean_segments['segment_type'] = BGN
rfi_segments = segment_labels.loc[
    segment_labels['label'].eq('bright_NBRFI'), ['global_index', 'segment_index']
].copy()
rfi_segments['segment_type'] = 'RFI'
annotated_segments = pd.concat([clean_segments, rfi_segments], ignore_index=True)
if annotated_segments['segment_index'].duplicated().any():
    raise ValueError('Annotated segment indices are not unique')

training_metadata = pd.read_csv(training_metadata_path)
if 'segment_index' not in training_metadata.columns:
    raise ValueError('Training metadata does not contain segment_index')
training_segment_indices = set(training_metadata['segment_index'].astype(int))
evaluation_segments = annotated_segments.loc[
    ~annotated_segments['segment_index'].isin(training_segment_indices)
].sort_values('segment_index').reset_index(drop=True)
if evaluation_segments.empty:
    raise ValueError('No annotated segments remain after training-segment exclusion')

for segment_index in evaluation_segments.loc[
    evaluation_segments['segment_type'].eq('RFI'), 'segment_index'
]:
    if get_rfi_block_id(int(segment_index)) is None:
        raise ValueError(f'No RFI block matches segment {segment_index}')

print('Training segments:', len(training_segment_indices))
print('Held-out segments:', len(evaluation_segments))
print(evaluation_segments['segment_type'].value_counts().sort_index())


Training segments: 12447
Held-out segments: 14209
segment_type
BGN    13174
RFI     1035
Name: count, dtype: int64


## Load saved models


In [5]:
class CNN1D_RFI_256_logits(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 64, kernel_size=7)
        self.pool1 = nn.MaxPool1d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=7)
        self.pool2 = nn.MaxPool1d(kernel_size=2, stride=2)
        self.conv3 = nn.Conv1d(128, 256, kernel_size=10)
        self.pool3 = nn.MaxPool1d(kernel_size=2, stride=2)
        self.flatten = nn.Flatten()
        self.dropout = nn.Dropout(p=0.5)
        self.fc1 = nn.Linear(6400, 256)
        self.fc2 = nn.Linear(256, 1)

    def forward(self, inputs):
        outputs = self.pool1(F.relu(self.conv1(inputs)))
        outputs = F.relu(self.conv2(outputs))
        outputs = F.pad(outputs, (0, 1))
        outputs = self.pool2(outputs)
        outputs = self.pool3(F.relu(self.conv3(outputs)))
        outputs = self.flatten(outputs)
        outputs = self.dropout(F.relu(self.fc1(outputs)))
        return self.fc2(outputs)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cnn_checkpoint = torch.load(cnn_checkpoint_path, map_location=device)
required_checkpoint_keys = {
    'epoch', 'model_state_dict', 'optimizer_state_dict',
    'validation_loss', 'validation_accuracy'
}
if set(cnn_checkpoint) != required_checkpoint_keys:
    raise ValueError('CNN checkpoint has an unexpected schema')

cnn_model = CNN1D_RFI_256_logits().to(device)
cnn_model.load_state_dict(cnn_checkpoint['model_state_dict'])
cnn_model.eval()
cnn_threshold = 0.5

statistical_model = joblib.load(statistical_model_path)
required_bundle_keys = {
    'model_name', 'feature_columns', 'threshold', 'pipeline',
    'validation_metrics', 'test_metrics'
}
if set(statistical_model) != required_bundle_keys:
    raise ValueError('Statistical-model bundle has an unexpected schema')
if len(statistical_model['feature_columns']) != 3:
    raise ValueError('Statistical-model bundle must use three features')

print('CNN device:', device.type)
print('CNN threshold:', cnn_threshold)
print('Statistical model:', statistical_model['model_name'])
print('Statistical threshold:', statistical_model['threshold'])


CNN device: cuda
CNN threshold: 0.5
Statistical model: MLP
Statistical threshold: 0.5800000000000001


## Define prediction functions


In [6]:
filterbank_file = your.Your(str(filterbank_file_path))
if filterbank_file.your_header.nchans != n_channels:
    raise ValueError('Filterbank channel count is not 256')


def load_segment(segment_index):
    segment = filterbank_file.get_data(
        nstart=segment_index * n_time_samples,
        nsamp=n_time_samples
    ).T
    if segment.shape != (n_channels, n_time_samples):
        raise ValueError(f'Unexpected segment shape for {segment_index}: {segment.shape}')
    return segment


def compute_stats(values, axis=None):
    values = np.asarray(values)
    mean = values.mean(axis=axis, keepdims=True)
    std = values.std(axis=axis, keepdims=True)
    centered = values - mean
    std_safe = np.where(std == 0, 1.0, std)
    skew = (centered ** 3).mean(axis=axis, keepdims=True) / (std_safe ** 3)
    kurt = (centered ** 4).mean(axis=axis, keepdims=True) / (std_safe ** 4) - 3.0
    return mean.squeeze(), std.squeeze(), skew.squeeze(), kurt.squeeze()


def normalize_per_channel(segment):
    channel_min = segment.min(axis=1, keepdims=True)
    channel_max = segment.max(axis=1, keepdims=True)
    channel_range = channel_max - channel_min
    channel_range[channel_range == 0] = 1.0
    return 256.0 * (segment - channel_min) / channel_range


def compute_ratio(channel_values, segment_value):
    if segment_value == 0:
        return np.full_like(channel_values, np.nan, dtype=float)
    return np.asarray(channel_values, dtype=float) / segment_value


def segment_features(segment):
    mean_o_segment, std_o_segment, skew_o_segment, kurt_o_segment = compute_stats(segment)
    normalized_segment = normalize_per_channel(segment)
    mean_n_segment, std_n_segment, skew_n_segment, kurt_n_segment = compute_stats(normalized_segment)
    mean_o, std_o, skew_o, kurt_o = compute_stats(segment, axis=1)
    mean_n, std_n, skew_n, kurt_n = compute_stats(normalized_segment, axis=1)
    return pd.DataFrame({
        'mean_o': mean_o, 'std_o': std_o, 'skew_o': skew_o, 'kurt_o': kurt_o,
        'mean_n': mean_n, 'std_n': std_n, 'skew_n': skew_n, 'kurt_n': kurt_n,
        'mean_o_ratio': compute_ratio(mean_o, mean_o_segment),
        'std_o_ratio': compute_ratio(std_o, std_o_segment),
        'skew_o_ratio': compute_ratio(skew_o, skew_o_segment),
        'kurt_o_ratio': compute_ratio(kurt_o, kurt_o_segment),
        'mean_n_ratio': compute_ratio(mean_n, mean_n_segment),
        'std_n_ratio': compute_ratio(std_n, std_n_segment),
        'skew_n_ratio': compute_ratio(skew_n, skew_n_segment),
        'kurt_n_ratio': compute_ratio(kurt_n, kurt_n_segment)
    })


def predict_cnn(segment):
    channels = np.asarray(segment, dtype=np.float32)
    channel_maxima = np.max(np.abs(channels), axis=1, keepdims=True)
    channel_maxima[channel_maxima < 1e-8] = 1.0
    inputs = torch.from_numpy(channels / channel_maxima).unsqueeze(1).to(device)
    with torch.no_grad():
        return torch.sigmoid(cnn_model(inputs)).squeeze(1).cpu().numpy()


def predict_statistical(segment):
    values = segment_features(segment).loc[:, statistical_model['feature_columns']]
    pipeline = statistical_model['pipeline']
    if hasattr(pipeline, 'predict_proba'):
        return pipeline.predict_proba(values)[:, 1]
    return 1.0 / (1.0 + np.exp(-pipeline.decision_function(values)))


## Define segment metrics


In [7]:
def segment_target(segment_index, segment_type):
    if segment_type == BGN:
        return np.zeros(n_channels, dtype=np.int64), BGN
    block_id = get_rfi_block_id(segment_index)
    rfi_mask = build_rfi_mask(get_rfi_channel_ranges(block_id))
    labels = np.where(rfi_mask, NBRFI, BGN_RFISEG)
    return rfi_mask.astype(np.int64), '|'.join(sorted(set(labels)))


def metric_row(model_name, threshold, segment_index, global_index, segment_type, target, labels_present, scores):
    prediction = (scores >= threshold).astype(np.int64)
    tn = int(((target == 0) & (prediction == 0)).sum())
    fp = int(((target == 0) & (prediction == 1)).sum())
    fn = int(((target == 1) & (prediction == 0)).sum())
    tp = int(((target == 1) & (prediction == 1)).sum())
    negative_count = tn + fp
    positive_count = fn + tp
    specificity = tn / negative_count if negative_count else np.nan
    row = {
        'model': model_name,
        'threshold': float(threshold),
        'global_index': int(global_index),
        'segment_index': int(segment_index),
        'segment_type': segment_type,
        'labels_present': labels_present,
        'n_channels': int(len(target)),
        'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp,
        'accuracy': float((tn + tp) / len(target)),
        'specificity': float(specificity),
        'false_positive_rate': float(fp / negative_count) if negative_count else np.nan,
        'mean_score': float(np.mean(scores)),
        'predicted_rfi_fraction': float(np.mean(prediction))
    }
    if positive_count:
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / positive_count
        row['precision'] = float(precision)
        row['recall'] = float(recall)
        row['f1'] = float(2 * precision * recall / (precision + recall)) if precision + recall else 0.0
        row['balanced_accuracy'] = float((recall + specificity) / 2)
    else:
        row['precision'] = np.nan
        row['recall'] = np.nan
        row['f1'] = np.nan
        row['balanced_accuracy'] = np.nan
    return row


def pooled_metrics(frame):
    tn, fp, fn, tp = (int(frame[column].sum()) for column in ('TN', 'FP', 'FN', 'TP'))
    total = tn + fp + fn + tp
    specificity = tn / (tn + fp) if tn + fp else np.nan
    if fn + tp:
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (fn + tp)
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        balanced_accuracy = (recall + specificity) / 2
    else:
        precision = recall = f1 = balanced_accuracy = np.nan
    return {
        'pooled_TN': tn, 'pooled_FP': fp, 'pooled_FN': fn, 'pooled_TP': tp,
        'pooled_accuracy': (tn + tp) / total,
        'pooled_specificity': specificity,
        'pooled_false_positive_rate': fp / (tn + fp) if tn + fp else np.nan,
        'pooled_precision': precision,
        'pooled_recall': recall,
        'pooled_f1': f1,
        'pooled_balanced_accuracy': balanced_accuracy
    }


## Run global test


In [8]:
metric_rows = []

for row in tqdm(evaluation_segments.itertuples(index=False), total=len(evaluation_segments), desc='Testing held-out segments'):
    segment_index = int(row.segment_index)
    segment = load_segment(segment_index)
    target, labels_present = segment_target(segment_index, row.segment_type)

    cnn_scores = predict_cnn(segment)
    metric_rows.append(metric_row(
        'CNN', cnn_threshold, segment_index, row.global_index,
        row.segment_type, target, labels_present, cnn_scores
    ))

    statistical_scores = predict_statistical(segment)
    metric_rows.append(metric_row(
        f"Statistical {statistical_model['model_name']}",
        statistical_model['threshold'], segment_index, row.global_index,
        row.segment_type, target, labels_present, statistical_scores
    ))

segment_metrics = pd.DataFrame(metric_rows).sort_values(['model', 'segment_index']).reset_index(drop=True)
if len(segment_metrics) != 2 * len(evaluation_segments):
    raise RuntimeError('Global-test metric count is invalid')
segment_metrics.to_csv(segment_metrics_path, index=False)
print('Segment metrics:', segment_metrics_path)


Testing held-out segments:   0%|          | 0/14209 [00:00<?, ?it/s]

Segment metrics: /u/akazantsev/ml_dl_for_rfi_mitigation/runs/punch_rfi_reproduction/global_test_segment_metrics.csv


## Summarize global test


In [9]:
metric_columns = [
    'accuracy', 'specificity', 'false_positive_rate', 'mean_score',
    'predicted_rfi_fraction', 'precision', 'recall', 'f1', 'balanced_accuracy'
]
summary_rows = []

for (model_name, segment_type), frame in segment_metrics.groupby(['model', 'segment_type'], sort=True):
    summary = {
        'model': model_name,
        'segment_type': segment_type,
        'threshold': float(frame['threshold'].iloc[0]),
        'n_segments': int(len(frame)),
        'n_channels': int(frame['n_channels'].sum())
    }
    for metric in metric_columns:
        summary[f'macro_{metric}'] = float(frame[metric].mean())
        summary[f'macro_{metric}_std'] = float(frame[metric].std(ddof=0))
    summary.update(pooled_metrics(frame))
    summary_rows.append(summary)

summary = pd.DataFrame(summary_rows).sort_values(['model', 'segment_type']).reset_index(drop=True)
summary.to_csv(summary_path, index=False)

for row in summary.itertuples(index=False):
    print(f'{row.model} | {row.segment_type} | segments: {row.n_segments} | threshold: {row.threshold:.2f}')
    print(f'  Macro accuracy: {row.macro_accuracy:.2%} ± {row.macro_accuracy_std:.2%}')
    print(f'  Macro specificity: {row.macro_specificity:.2%} ± {row.macro_specificity_std:.2%}')
    print(f'  Macro false-positive rate: {row.macro_false_positive_rate:.2%} ± {row.macro_false_positive_rate_std:.2%}')
    print(f'  Macro predicted-RFI fraction: {row.macro_predicted_rfi_fraction:.2%} ± {row.macro_predicted_rfi_fraction_std:.2%}')
    if row.segment_type == 'RFI':
        print(f'  Macro precision: {row.macro_precision:.2%} ± {row.macro_precision_std:.2%}')
        print(f'  Macro recall: {row.macro_recall:.2%} ± {row.macro_recall_std:.2%}')
        print(f'  Macro F1: {row.macro_f1:.2%} ± {row.macro_f1_std:.2%}')
        print(f'  Macro balanced accuracy: {row.macro_balanced_accuracy:.2%} ± {row.macro_balanced_accuracy_std:.2%}')

print('Summary:', summary_path)
display(summary)


CNN | BGN | segments: 13174 | threshold: 0.50
  Macro accuracy: 97.19% ± 1.07%
  Macro specificity: 97.19% ± 1.07%
  Macro false-positive rate: 2.81% ± 1.07%
  Macro predicted-RFI fraction: 2.81% ± 1.07%
CNN | RFI | segments: 1035 | threshold: 0.50
  Macro accuracy: 93.96% ± 1.94%
  Macro specificity: 94.59% ± 2.52%
  Macro false-positive rate: 5.41% ± 2.52%
  Macro predicted-RFI fraction: 9.59% ± 3.29%
  Macro precision: 46.91% ± 14.80%
  Macro recall: 83.38% ± 19.24%
  Macro F1: 57.77% ± 12.93%
  Macro balanced accuracy: 88.98% ± 8.96%
Statistical MLP | BGN | segments: 13174 | threshold: 0.58
  Macro accuracy: 98.74% ± 0.73%
  Macro specificity: 98.74% ± 0.73%
  Macro false-positive rate: 1.26% ± 0.73%
  Macro predicted-RFI fraction: 1.26% ± 0.73%
Statistical MLP | RFI | segments: 1035 | threshold: 0.58
  Macro accuracy: 93.13% ± 6.42%
  Macro specificity: 93.29% ± 7.06%
  Macro false-positive rate: 6.71% ± 7.06%
  Macro predicted-RFI fraction: 11.21% ± 7.12%
  Macro precision: 49.24

,model,segment_type,threshold,n_segments,n_channels,macro_accuracy,macro_accuracy_std,macro_specificity,macro_specificity_std,macro_false_positive_rate,...,pooled_FP,pooled_FN,pooled_TP,pooled_accuracy,pooled_specificity,pooled_false_positive_rate,pooled_precision,pooled_recall,pooled_f1,pooled_balanced_accuracy
0,CNN,BGN,0.50,13174,3372544,0.971879,0.010693,0.971879,0.010693,0.028121,...,94838,0,0,0.971879,0.971879,0.028121,NaN,NaN,NaN,NaN
1,CNN,RFI,0.50,1035,264960,0.939647,0.019429,0.945860,0.025238,0.054140,...,13601,2390,11819,0.939647,0.945759,0.054241,0.464949,0.831797,0.596482,0.888778
2,Statistical MLP,BGN,0.58,13174,3372544,0.987385,0.007262,0.987385,0.007262,0.012615,...,42546,0,0,0.987385,0.987385,0.012615,NaN,NaN,NaN,NaN
3,Statistical MLP,RFI,0.58,1035,264960,0.931329,0.064158,0.932864,0.070644,0.067136,...,16847,1348,12861,0.931329,0.932814,0.067186,0.432914,0.905131,0.585696,0.918972
